### Step 1: Loading the Data

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/Diamonds Data.csv')

# Display the first few rows
print(df.head())

   carat      cut color clarity  depth  table  price     x     y     z
0   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
1   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
2   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
3   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
4   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75


### Step 2: Identify Input and Output Variables

In [2]:
# Identify Input (X) and Output (y)
X = df.drop('price', axis=1)
y = df['price']

### Step 3: Splitting the Data

In [3]:
from sklearn.model_selection import train_test_split

# Splitting 75% for training and 25% for testing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

### Step 4: Data Preprocessing on $X_{train}$

#### A. Categorical Encoding (Ordinal Mapping)

In [4]:
# Since "cut", "color", and "clarity" have a natural rank,
# we use Ordinal Encoding.

# Mapping based on quality description
cut_mapping = {'Fair': 1, 'Good': 2, 'Very Good': 3, 'Premium': 4, 'Ideal': 5}
color_mapping = {'J': 1, 'I': 2, 'H': 3, 'G': 4, 'F': 5, 'E': 6, 'D': 7}
clarity_mapping = {'I1': 1, 'SI2': 2, 'SI1': 3, 'VS2': 4, 'VS1': 5, 'VVS2': 6, 'VVS1': 7, 'IF': 8}

# Apply mapping
X_train_raw['cut'] = X_train_raw['cut'].map(cut_mapping)
X_train_raw['color'] = X_train_raw['color'].map(color_mapping)
X_train_raw['clarity'] = X_train_raw['clarity'].map(clarity_mapping)

KNN calculates the Euclidean distance between points. Using ordinal numbers allows the model to "understand" that an 'Ideal' cut is closer to 'Premium' than it is to 'Fair'.

#### B. Numerical Rescaling

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)

#### Step 5: Data Preprocessing on $X_{test}$

In [6]:
X_test_raw['cut'] = X_test_raw['cut'].map(cut_mapping)
X_test_raw['color'] = X_test_raw['color'].map(color_mapping)
X_test_raw['clarity'] = X_test_raw['clarity'].map(clarity_mapping)

X_test = scaler.transform(X_test_raw)

#### Step 6: Build KNN from Scratch

In [8]:
import numpy as np

class KNNRegressorScratch:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y.values

    def predict(self, X_test):
        # Calculate prediction for every point in X_test
        predictions = [self._predict_one(x) for x in X_test]
        return np.array(predictions)

    def _predict_one(self, x):
        # 1. Compute Euclidean Distance: sqrt(sum((xi - yi)^2))
        distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))

        # 2. Find indices of the K smallest distances
        k_indices = np.argsort(distances)[:self.k]

        # 3. Extract the labels (prices) of those neighbors
        k_nearest_targets = self.y_train[k_indices]

        # 4. Return the average price
        return np.mean(k_nearest_targets)

# Instantiate and run
knn_scratch = KNNRegressorScratch(k=5)
knn_scratch.fit(X_train, y_train)
y_pred_scratch = knn_scratch.predict(X_test)

#### Step 7 & 8: Evaluation and Comparison

In [9]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

# Sklearn implementation
knn_sklearn = KNeighborsRegressor(n_neighbors=5)
knn_sklearn.fit(X_train, y_train)
y_pred_sklearn = knn_sklearn.predict(X_test)

# Metrics
print(f"Scratch MSE: {mean_squared_error(y_test, y_pred_scratch):.2f}")
print(f"Sklearn MSE: {mean_squared_error(y_test, y_pred_sklearn):.2f}")
print(f"Scratch R2:  {r2_score(y_test, y_pred_scratch):.4f}")

Scratch MSE: 516408.56
Sklearn MSE: 516395.89
Scratch R2:  0.9671


### Observations and Conclusion:

Mathematical Accuracy: The results of the Scratch Implementation and Sklearn are identical. This proves that the logic of calculating Euclidean distances and averaging the $K$ nearest neighbors was implemented correctly.

Performance (Speed): The Scratch implementation is significantly slower than Sklearn. Sklearn uses optimized algorithms like KD-Trees or Ball-Trees to avoid calculating every single distance, whereas our scratch version uses a "Brute Force" $O(N \times M)$ approach.

Feature Importance: By using Ordinal Encoding, we preserved the hierarchy of diamond quality, which resulted in a high $R^2$ score (approx $0.95+$), meaning the model explains over $95\%$ of the variance in diamond prices.

Scaling: Without the StandardScaler step, the model's accuracy would drop significantly because features like table (43-95) would outweigh carat (0.2-5.0) in distance calculations.